Is DEI / AI discourse in Information Technology firms distinctive, once we control for general market-wide language inflation?

In [14]:
# What this cell does:
# End-to-end DEI + AI *Adoption Signal* (share of firm-years with ≥1 mention), v2
# -> Load AI_DEI_Lexicon.yml and extract AI + DEI terms
# -> Load filings, derive year
# -> Collapse sections -> firm-year document (dtype-safe, works if text is Utf8 OR list[str])
# -> Build robust regex (handles A.I., D.E.I., hyphenation, line breaks)
# -> Compute capped-IDF weights across firm-year docs (kept for later intensity variants)
# -> Score presence (0/1) and capped weighted hits (cap per term) per 1,000 tokens
# -> Aggregate by year: All sectors vs IT
# -> Plot + save PNGs reproducibly (versioned, non-overwriting)
# NOTE: This cell writes TWO plots only: adoption (DEI, AI). No score plot overwrite.

from __future__ import annotations

from pathlib import Path
from datetime import datetime
import os
import re
import math
import yaml

import polars as pl
import matplotlib.pyplot as plt


# --------------------
# Config
# --------------------
PARQUET_FILE = Path("./spy_10k_2015_present.parquet")
LEXICON_FILE = Path("./AI_DEI_Lexicon.yml")

YEAR_MIN, YEAR_MAX = 2015, 2025
IT_SECTOR_NAME = "Information Technology"

TEXT_COL = "text"
SECTOR_COL = "gics_sector"
DATE_COL = "filing_date"
PERIOD_COL = "filing_period"
FIRM_COL = "ticker"

IDF_CAP = 8.0
TERM_HIT_CAP = 3

# What this does: stable outputs root + versioned, non-overwriting filenames
OUT_DIR = Path("./outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

STAMP = datetime.now().strftime("%Y%m%d_%H%M")
RUN_LABEL = "v2"  # "token-weighted" or "v2"

OUT_DEI_ADOPTION = OUT_DIR / f"it_vs_all_DEI_adoption_{RUN_LABEL}_{STAMP}.png"
OUT_AI_ADOPTION  = OUT_DIR / f"it_vs_all_AI_adoption_{RUN_LABEL}_{STAMP}.png"

print("Writing outputs to:")
print("DEI adoption:", OUT_DEI_ADOPTION)
print("AI  adoption:", OUT_AI_ADOPTION)


# --------------------
# Helpers
# --------------------
def _clean_term(t: str) -> str:
    t = t.strip()
    t = re.sub(r"\s+", " ", t)
    t = t.strip(" ,;:.")
    return t

def _extract_strings(node):
    out = []
    if node is None:
        return out
    if isinstance(node, str):
        out.append(node)
    elif isinstance(node, list):
        for x in node:
            out.extend(_extract_strings(x))
    elif isinstance(node, dict):
        for v in node.values():
            out.extend(_extract_strings(v))
    return out

def _find_terms_any(obj, want: str) -> list[str]:
    subkeys = {"terms", "phrases", "patterns", "lexicon", "keywords"}

    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).lower() == want.lower():
                terms = [_clean_term(x) for x in _extract_strings(v) if _clean_term(x)]
                if terms:
                    return sorted(set(terms))
                if isinstance(v, dict):
                    for sk in subkeys:
                        if sk in v:
                            terms = [_clean_term(x) for x in _extract_strings(v[sk]) if _clean_term(x)]
                            if terms:
                                return sorted(set(terms))
        for v in obj.values():
            found = _find_terms_any(v, want)
            if found:
                return found

    if isinstance(obj, list):
        for x in obj:
            found = _find_terms_any(x, want)
            if found:
                return found

    return []

def _regex_for_term(t: str) -> str:
    # What this does: robust matching for filings:
    # - acronyms with dots/spaces: A.I., D E I, D.E.I.
    # - phrase hyphenation/linebreak: artificial-\nintelligence
    # - unicode hyphens
    s = _clean_term(t)

    # Acronyms like AI/DEI/EEO: allow dot/space/hyphen between letters
    if re.fullmatch(r"[A-Za-z]{2,6}", s):
        mid = r"[\.\s\-]*"
        letters = list(s)
        inner = mid.join(map(re.escape, letters))
        return rf"(?i)\b{inner}\b"

    # Phrase: allow whitespace OR hyphen (incl unicode hyphens) between tokens
    parts = [re.escape(p) for p in s.split()]
    joiner = r"(?:\s+|\-|\u2010|\u2011|\u2012|\u2013|\u2014)+"
    phrase = joiner.join(parts)
    return rf"(?i)\b{phrase}\b"

def _idf(df_term: int, N: int, cap: float) -> float:
    val = math.log((N + 1) / (df_term + 1)) + 1.0
    return min(val, cap)

def _idf_weights(df_docs: pl.DataFrame, terms: list[str], cap: float) -> dict[str, float]:
    N_docs = df_docs.height
    weights = {}
    for t in terms:
        rt = _regex_for_term(t)
        df_term = df_docs.select(pl.col(TEXT_COL).str.contains(rt).sum().alias("df")).item()
        weights[t] = _idf(int(df_term), int(N_docs), cap)
    return weights

def _presence_expr(terms: list[str]) -> pl.Expr:
    exprs = [pl.col(TEXT_COL).str.contains(_regex_for_term(t)) for t in terms]
    if not exprs:
        return pl.lit(0).cast(pl.Int64)
    out = exprs[0]
    for e in exprs[1:]:
        out = out | e
    return out.cast(pl.Int64)

def _capped_weighted_hits_expr(terms: list[str], weights: dict[str, float], cap_per_term: int) -> pl.Expr:
    exprs = []
    cap_lit = pl.lit(int(cap_per_term))
    for t in terms:
        rt = _regex_for_term(t)
        c = pl.col(TEXT_COL).str.count_matches(rt)
        c_cap = pl.min_horizontal(c, cap_lit)
        exprs.append(c_cap * pl.lit(float(weights[t])))
    if not exprs:
        return pl.lit(0.0)
    out = exprs[0]
    for e in exprs[1:]:
        out = out + e
    return out

def _save_lineplot(years, y_all, y_it, title, ylabel, out_path: Path):
    plt.figure(figsize=(12, 8))
    plt.plot(years, y_all, label="All sectors")
    if any(v is not None for v in y_it):
        plt.plot(years, y_it, label="Information Technology")
    plt.title(title)
    plt.xlabel("Year")
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


# --------------------
# Lexicon
# --------------------
if not LEXICON_FILE.exists():
    raise FileNotFoundError("AI_DEI_Lexicon.yml not found in the working directory.")

lex = yaml.safe_load(LEXICON_FILE.read_text(encoding="utf-8"))

AI_TERMS  = _find_terms_any(lex, "AI")
DEI_TERMS = _find_terms_any(lex, "DEI")

print("AI terms:", len(AI_TERMS), "DEI terms:", len(DEI_TERMS))
print("AI sample:", AI_TERMS[:20])
print("DEI sample:", DEI_TERMS[:20])

if not AI_TERMS or not DEI_TERMS:
    raise ValueError("Lexicon extraction failed — no terms found.")


# --------------------
# Load + year
# --------------------
if not PARQUET_FILE.exists():
    raise FileNotFoundError("spy_10k_2015_present.parquet not found in the working directory.")

df = pl.read_parquet(PARQUET_FILE)

df = df.with_columns(
    pl.coalesce([
        pl.col(DATE_COL).cast(pl.Date, strict=False).dt.year(),
        pl.col(PERIOD_COL).cast(pl.Utf8).str.extract(r"(\d{4})", 1).cast(pl.Int64),
    ]).alias("year")
).filter(
    pl.col("year").is_between(YEAR_MIN, YEAR_MAX)
)


# --------------------
# Collapse sections -> firm-year doc (dtype-safe)
# --------------------
text_dtype = df.schema.get(TEXT_COL)

if isinstance(text_dtype, pl.List):
    df_fy = (
        df.group_by([FIRM_COL, "year", SECTOR_COL])
          .agg(pl.concat_list(pl.col(TEXT_COL)).flatten().alias("_chunks"))
          .with_columns(pl.col("_chunks").list.join("\n").alias(TEXT_COL))
          .drop("_chunks")
    )
else:
    df_fy = (
        df.group_by([FIRM_COL, "year", SECTOR_COL])
          .agg(pl.col(TEXT_COL).cast(pl.Utf8, strict=False).alias("_chunks"))
          .with_columns(pl.concat_list(pl.col("_chunks")).list.join("\n").alias(TEXT_COL))
          .drop("_chunks")
    )

# Compute doc_length (kept for later intensity variants)
df_fy = df_fy.with_columns(
    pl.col(TEXT_COL).str.split(by=r"\s+").list.len().cast(pl.Int64).alias("doc_length")
).filter(pl.col("doc_length") > 0)

it_mask = pl.col(SECTOR_COL).cast(pl.Utf8).str.contains(IT_SECTOR_NAME, literal=True)

print("Firm-years (all):", df_fy.height)
print("Firm-years (IT):", df_fy.filter(it_mask).height)


# --------------------
# Weights (kept for later, but computed here so intensity cols exist)
# --------------------
w_dei = _idf_weights(df_fy, DEI_TERMS, IDF_CAP)
w_ai  = _idf_weights(df_fy, AI_TERMS,  IDF_CAP)


# --------------------
# Score (presence + capped weighted hits, per 1,000 tokens)
# --------------------
scored = df_fy.with_columns(
    _presence_expr(DEI_TERMS).alias("dei_presence"),
    _presence_expr(AI_TERMS).alias("ai_presence"),
    _capped_weighted_hits_expr(DEI_TERMS, w_dei, TERM_HIT_CAP).alias("dei_hits_w_cap"),
    _capped_weighted_hits_expr(AI_TERMS,  w_ai,  TERM_HIT_CAP).alias("ai_hits_w_cap"),
).with_columns(
    (pl.col("dei_hits_w_cap") / pl.col("doc_length") * 1000.0).alias("dei_intensity"),
    (pl.col("ai_hits_w_cap")  / pl.col("doc_length") * 1000.0).alias("ai_intensity"),
)

print(
    scored.select(
        pl.mean("dei_presence").alias("mean_dei_presence"),
        pl.mean("ai_presence").alias("mean_ai_presence"),
        pl.mean("dei_intensity").alias("mean_dei_intensity"),
        pl.mean("ai_intensity").alias("mean_ai_intensity"),
    )
)


# --------------------
# Aggregate by year (ADOPTION) + plot with corrected naming
# --------------------
year_all = (
    scored.group_by("year")
    .agg(
        pl.mean("dei_presence").alias("dei_adoption_all"),
        pl.mean("ai_presence").alias("ai_adoption_all"),
    )
    .sort("year")
)

year_it = (
    scored.filter(it_mask)
    .group_by("year")
    .agg(
        pl.mean("dei_presence").alias("dei_adoption_it"),
        pl.mean("ai_presence").alias("ai_adoption_it"),
    )
    .sort("year")
)

ts = year_all.join(year_it, on="year", how="left").sort("year")

_save_lineplot(
    ts["year"].to_list(),
    ts["dei_adoption_all"].to_list(),
    ts["dei_adoption_it"].to_list(),
    "DEI Disclosure Adoption Signal (share of firm-years with ≥1 mention): IT vs All sectors",
    "Adoption rate (share of firm-years; 0–1)",
    OUT_DEI_ADOPTION,
)

_save_lineplot(
    ts["year"].to_list(),
    ts["ai_adoption_all"].to_list(),
    ts["ai_adoption_it"].to_list(),
    "AI Disclosure Adoption Signal (share of firm-years with ≥1 mention): IT vs All sectors",
    "Adoption rate (share of firm-years; 0–1)",
    OUT_AI_ADOPTION,
)

OUT_DEI_ADOPTION, OUT_AI_ADOPTION


Writing outputs to:
DEI adoption: outputs\it_vs_all_DEI_adoption_v2_20260204_2217.png
AI  adoption: outputs\it_vs_all_AI_adoption_v2_20260204_2217.png
AI terms: 95 DEI terms: 99
AI sample: ['(?i)\\ba\\.i\\.\\b', '(?i)\\baccountability\\b', '(?i)\\bagents?\\b', '(?i)\\bai[-\\s]+ethics\\b', '(?i)\\bai[-\\s]+fairness\\b', '(?i)\\bai[-\\s]+risk\\b', '(?i)\\bai[-\\s]+safety\\b', '(?i)\\bai\\b', '(?i)\\balgorithmic[-\\s]+bias\\b', '(?i)\\balgorithmic\\b', '(?i)\\balgorithms?\\b', '(?i)\\banomaly[-\\s]+detection\\b', '(?i)\\bartificial[-\\s]+intelligence\\b', '(?i)\\bautomation\\b', '(?i)\\bautonomous[-\\s]+agents?\\b', '(?i)\\bautonomous[-\\s]+systems?\\b', '(?i)\\bbias\\b', '(?i)\\bchatgpt\\b', '(?i)\\bclassification\\b', '(?i)\\bclustering\\b']
DEI sample: ['(?i)\\b(equal|fair)[-\\s]+treatment\\b', '(?i)\\b(equal|gender)[-\\s]+pay\\b', '(?i)\\b(first[-\\s]+nations)\\b', '(?i)\\b(inclusive|inclusivity)\\b', '(?i)\\b(unconscious|implicit)[-\\s]+bias\\b', '(?i)\\baccessibility\\b', '(?i)\\bac

(WindowsPath('outputs/it_vs_all_DEI_adoption_v2_20260204_2217.png'),
 WindowsPath('outputs/it_vs_all_AI_adoption_v2_20260204_2217.png'))

In [15]:
# What this does: measure which regex patterns ever match anything in your corpus
def _coverage_audit(df_docs: pl.DataFrame, patterns: list[str], label: str, top_k: int = 25):
    rows = []
    N = df_docs.height
    for pat in patterns:
        try:
            hits = df_docs.select(pl.col(TEXT_COL).str.contains(pat).sum().alias("n")).item()
            rows.append((pat, int(hits), float(hits) / float(N)))
        except Exception as e:
            rows.append((pat, -1, 0.0))
    out = pl.DataFrame(rows, schema=["pattern", "n_docs", "share_docs"])
    print(f"\n=== COVERAGE AUDIT: {label} (N_docs={N}) ===")
    print("Patterns with zero matches:", int((out["n_docs"] == 0).sum()))
    print("Patterns with errors:", int((out["n_docs"] < 0).sum()))
    print("\nTop matching patterns:")
    print(out.sort("n_docs", descending=True).head(top_k))
    return out

AI_AUDIT  = _coverage_audit(df_fy, AI_TERMS,  "AI")
DEI_AUDIT = _coverage_audit(df_fy, DEI_TERMS, "DEI")


C:\Users\ddddd\AppData\Local\Temp\ipykernel_19012\3585048984.py:11: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  out = pl.DataFrame(rows, schema=["pattern", "n_docs", "share_docs"])



=== COVERAGE AUDIT: AI (N_docs=4925) ===
Patterns with zero matches: 26
Patterns with errors: 0

Top matching patterns:
shape: (25, 3)
┌─────────────────────────────────┬────────┬────────────┐
│ pattern                         ┆ n_docs ┆ share_docs │
│ ---                             ┆ ---    ┆ ---        │
│ str                             ┆ i64    ┆ f64        │
╞═════════════════════════════════╪════════╪════════════╡
│ (?i)\bagents?\b                 ┆ 2912   ┆ 0.591269   │
│ (?i)\btransparency\b            ┆ 1724   ┆ 0.350051   │
│ (?i)\bautomation\b              ┆ 1480   ┆ 0.300508   │
│ (?i)\baccountability\b          ┆ 1416   ┆ 0.287513   │
│ (?i)\bartificial[-\s]+intellig… ┆ 1312   ┆ 0.266396   │
│ …                               ┆ …      ┆ …          │
│ (?i)\bmodel[-\s]+risk\b         ┆ 71     ┆ 0.014416   │
│ (?i)\bdata[-\s]+scientists?\b   ┆ 63     ┆ 0.012792   │
│ (?i)\bcomputer[-\s]+assisted\b  ┆ 57     ┆ 0.011574   │
│ (?i)\bdeep[-\s]+learning\b      ┆ 57     ┆ 0.01157

AI is a relatively new addition to S&P 500 disclosure. We decide to optimise by focusing our lexicon.

In [16]:
# What this cell does:
# AI Disclosure Signal v2 (anchor-gated; ultra-small lexicon)
# -> Load filings, derive year
# -> Collapse sections -> firm-year document (dtype-safe; Utf8 OR list[str])
# -> Define ultra-small AI anchors + (optional) tiny AI-adjacent set
# -> Score:
#       ai_adoption = 1 if any anchor hits, else 0
#       ai_intensity = (anchor_hits_cap + gated_adjacent_hits_cap) per 1,000 tokens
# -> Aggregate by year: All sectors vs IT
# -> Plot + save PNGs reproducibly (versioned, non-overwriting)

from __future__ import annotations

from pathlib import Path
from datetime import datetime
import re
import os

import polars as pl
import matplotlib.pyplot as plt


# --------------------
# Config
# --------------------
PARQUET_FILE = Path("./spy_10k_2015_present.parquet")

YEAR_MIN, YEAR_MAX = 2015, 2025
IT_SECTOR_NAME = "Information Technology"

TEXT_COL = "text"
SECTOR_COL = "gics_sector"
DATE_COL = "filing_date"
PERIOD_COL = "filing_period"
FIRM_COL = "ticker"

# What this does: cap per-pattern frequency so boilerplate repeats don't dominate
HIT_CAP = 3

# Outputs (versioned; never overwrites)
OUT_DIR = Path("./outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
STAMP = datetime.now().strftime("%Y%m%d_%H%M")
RUN_LABEL = "ai_ultrasmall_v2"

OUT_AI_ADOPTION  = OUT_DIR / f"it_vs_all_AI_adoption_{RUN_LABEL}_{STAMP}.png"
OUT_AI_INTENSITY = OUT_DIR / f"it_vs_all_AI_intensity_{RUN_LABEL}_{STAMP}.png"

print("Writing outputs to:")
print("AI adoption :", OUT_AI_ADOPTION)
print("AI intensity:", OUT_AI_INTENSITY)


# --------------------
# Ultra-small AI lexicon (regex patterns, already compiled as strings)
# --------------------
# What this does: only unambiguous AI language. No "agents", no "automation", no "transparency", etc.
AI_ANCHOR_PATS = [
    r"(?i)\bartificial[-\s]+intelligence\b",
    r"(?i)\bmachine[-\s]+learning\b",
    r"(?i)\bdeep[-\s]+learning\b",
    r"(?i)\bneural[-\s]+networks?\b",
    r"(?i)\bnatural[-\s]+language[-\s]+processing\b",
    r"(?i)\bcomputer[-\s]+vision\b",
    r"(?i)\breinforcement[-\s]+learning\b",
    r"(?i)\bgenerative[-\s]+ai\b",
    r"(?i)\blarge[-\s]+language[-\s]+models?\b",
    r"(?i)\bllm(s)?\b",
]

# What this does: optional, still fairly specific, but gated behind anchors
AI_ADJACENT_PATS = [
    r"(?i)\bmodel[-\s]+training\b",
    r"(?i)\bmodel[-\s]+inference\b",
    r"(?i)\btraining[-\s]+data\b",
    r"(?i)\btraining[-\s]+set\b",
    r"(?i)\btesting[-\s]+set\b",
    r"(?i)\bvalidation[-\s]+set\b",
    r"(?i)\bmodel[-\s]+deployment\b",
]

print("AI anchors:", len(AI_ANCHOR_PATS), "| AI adjacent (gated):", len(AI_ADJACENT_PATS))


# --------------------
# Helpers
# --------------------
def _save_lineplot(years, y_all, y_it, title, ylabel, out_path: Path):
    plt.figure(figsize=(12, 8))
    plt.plot(years, y_all, label="All sectors")
    if any(v is not None for v in y_it):
        plt.plot(years, y_it, label="Information Technology")
    plt.title(title)
    plt.xlabel("Year")
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

def _presence_expr_patterns(patterns: list[str]) -> pl.Expr:
    exprs = [pl.col(TEXT_COL).str.contains(p) for p in patterns]
    if not exprs:
        return pl.lit(0).cast(pl.Int64)
    out = exprs[0]
    for e in exprs[1:]:
        out = out | e
    return out.cast(pl.Int64)

def _capped_hits_expr_patterns(patterns: list[str], cap_per_pat: int) -> pl.Expr:
    cap_lit = pl.lit(int(cap_per_pat))
    exprs = [pl.min_horizontal(pl.col(TEXT_COL).str.count_matches(p), cap_lit) for p in patterns]
    if not exprs:
        return pl.lit(0).cast(pl.Int64)
    out = exprs[0]
    for e in exprs[1:]:
        out = out + e
    return out.cast(pl.Int64)


# --------------------
# Load + year
# --------------------
if not PARQUET_FILE.exists():
    raise FileNotFoundError("spy_10k_2015_present.parquet not found in the working directory.")

df = pl.read_parquet(PARQUET_FILE)

df = df.with_columns(
    pl.coalesce([
        pl.col(DATE_COL).cast(pl.Date, strict=False).dt.year(),
        pl.col(PERIOD_COL).cast(pl.Utf8).str.extract(r"(\d{4})", 1).cast(pl.Int64),
    ]).alias("year")
).filter(
    pl.col("year").is_between(YEAR_MIN, YEAR_MAX)
)


# --------------------
# Collapse sections -> firm-year doc (dtype-safe)
# --------------------
text_dtype = df.schema.get(TEXT_COL)

if isinstance(text_dtype, pl.List):
    df_fy = (
        df.group_by([FIRM_COL, "year", SECTOR_COL])
          .agg(pl.concat_list(pl.col(TEXT_COL)).flatten().alias("_chunks"))
          .with_columns(pl.col("_chunks").list.join("\n").alias(TEXT_COL))
          .drop("_chunks")
    )
else:
    df_fy = (
        df.group_by([FIRM_COL, "year", SECTOR_COL])
          .agg(pl.col(TEXT_COL).cast(pl.Utf8, strict=False).alias("_chunks"))
          .with_columns(pl.concat_list(pl.col("_chunks")).list.join("\n").alias(TEXT_COL))
          .drop("_chunks")
    )

df_fy = df_fy.with_columns(
    pl.col(TEXT_COL).str.split(by=r"\s+").list.len().cast(pl.Int64).alias("doc_length")
).filter(pl.col("doc_length") > 0)

it_mask = pl.col(SECTOR_COL).cast(pl.Utf8).str.contains(IT_SECTOR_NAME, literal=True)

print("Firm-years (all):", df_fy.height)
print("Firm-years (IT):", df_fy.filter(it_mask).height)


# --------------------
# Score (anchor-gated)
# --------------------
ai_anchor_presence = _presence_expr_patterns(AI_ANCHOR_PATS)
ai_anchor_hits_cap = _capped_hits_expr_patterns(AI_ANCHOR_PATS, HIT_CAP)
ai_adj_hits_cap = _capped_hits_expr_patterns(AI_ADJACENT_PATS, HIT_CAP)

scored = df_fy.with_columns(
    ai_anchor_presence.alias("ai_adoption"),
    ai_anchor_hits_cap.alias("ai_anchor_hits_cap"),
    pl.when(ai_anchor_presence == 1)
      .then(ai_adj_hits_cap)
      .otherwise(pl.lit(0))
      .alias("ai_adj_hits_cap_gated"),
).with_columns(
    ((pl.col("ai_anchor_hits_cap") + pl.col("ai_adj_hits_cap_gated")) / pl.col("doc_length") * 1000.0)
    .alias("ai_intensity")
)

print(
    scored.select(
        pl.sum("ai_adoption").alias("sum_ai_adopters"),
        pl.mean("ai_adoption").alias("mean_ai_adoption"),
        pl.mean("ai_intensity").alias("mean_ai_intensity"),
    )
)


# --------------------
# Aggregate by year + plot
# --------------------
year_all = (
    scored.group_by("year")
    .agg(
        pl.mean("ai_adoption").alias("ai_adoption_all"),
        pl.mean("ai_intensity").alias("ai_intensity_all"),
    )
    .sort("year")
)

year_it = (
    scored.filter(it_mask)
    .group_by("year")
    .agg(
        pl.mean("ai_adoption").alias("ai_adoption_it"),
        pl.mean("ai_intensity").alias("ai_intensity_it"),
    )
    .sort("year")
)

ts = year_all.join(year_it, on="year", how="left").sort("year")

_save_lineplot(
    ts["year"].to_list(),
    ts["ai_adoption_all"].to_list(),
    ts["ai_adoption_it"].to_list(),
    "AI Disclosure Adoption Signal (ultra-small anchors; gated): IT vs All sectors",
    "Adoption rate (share of firm-years; 0–1)",
    OUT_AI_ADOPTION,
)

_save_lineplot(
    ts["year"].to_list(),
    ts["ai_intensity_all"].to_list(),
    ts["ai_intensity_it"].to_list(),
    "AI Disclosure Intensity (anchors + gated adjacent; capped; per 1,000 tokens): IT vs All sectors",
    "Capped hits per 1,000 tokens",
    OUT_AI_INTENSITY,
)

OUT_AI_ADOPTION, OUT_AI_INTENSITY


Writing outputs to:
AI adoption : outputs\it_vs_all_AI_adoption_ai_ultrasmall_v2_20260204_2223.png
AI intensity: outputs\it_vs_all_AI_intensity_ai_ultrasmall_v2_20260204_2223.png
AI anchors: 10 | AI adjacent (gated): 7
Firm-years (all): 4925
Firm-years (IT): 676
shape: (1, 3)
┌─────────────────┬──────────────────┬───────────────────┐
│ sum_ai_adopters ┆ mean_ai_adoption ┆ mean_ai_intensity │
│ ---             ┆ ---              ┆ ---               │
│ i64             ┆ f64              ┆ f64               │
╞═════════════════╪══════════════════╪═══════════════════╡
│ 1439            ┆ 0.292183         ┆ 1040.609137       │
└─────────────────┴──────────────────┴───────────────────┘


(WindowsPath('outputs/it_vs_all_AI_adoption_ai_ultrasmall_v2_20260204_2223.png'),
 WindowsPath('outputs/it_vs_all_AI_intensity_ai_ultrasmall_v2_20260204_2223.png'))

This proved successful. We do the same for DEI: